In [3]:
from azure.ai.ml import MLClient
from azure.ai.ml.entities import ManagedOnlineEndpoint, ManagedOnlineDeployment
from azure.identity import DefaultAzureCredential

SUBSCRIPTION_ID = "48b470e1-2c96-449b-8d68-d475b32d0a68"
RESOURCE_GROUP = "rg-mlab1-Zohaib-hassan"
WORKSPACE = "mlw-lab1-Zohaib-hassan"

ml_client = MLClient(DefaultAzureCredential(), SUBSCRIPTION_ID, RESOURCE_GROUP, WORKSPACE)
print("Connected to:", WORKSPACE)

Connected to: mlw-lab1-Zohaib-hassan


In [4]:
ENDPOINT_NAME = "knn-maint-zohaibhassan2"

endpoint = ManagedOnlineEndpoint(
    name=ENDPOINT_NAME,
    description="KNN predictive maintenance (notebook model)",
    auth_mode="key",
)
ml_client.online_endpoints.begin_create_or_update(endpoint).result()
print("Endpoint created:", ENDPOINT_NAME)

/anaconda/envs/azureml_py310_sdkv2/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Endpoint created: knn-maint-zohaibhassan2


In [1]:
import os, json, shutil
import pandas as pd, numpy, sklearn
import mlflow, mlflow.sklearn
from mlflow.models import infer_signature

if int(mlflow.__version__.split(".")[0]) >= 3:
    raise RuntimeError(
        "MLflow is still 3.x. Run: %pip install 'mlflow<3' azureml-mlflow --quiet "
        "in its own cell, restart the kernel, then run this cell again."
    )
print("mlflow version:", mlflow.__version__)

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier
from azure.ai.ml import MLClient
from azure.ai.ml.constants import AssetTypes
from azure.ai.ml.entities import Model, ManagedOnlineEndpoint, ManagedOnlineDeployment
from azure.identity import DefaultAzureCredential

ml_client = MLClient(
    DefaultAzureCredential(),
    "48b470e1-2c96-449b-8d68-d475b32d0a68",
    "rg-mlab1-Zohaib-hassan",
    "mlw-lab1-Zohaib-hassan",
)
ENDPOINT_NAME = "knn-maint-zohaibhassan2"

# 1. Rebuild the KNN model
repo = os.path.expanduser("~/cloudfiles/code/Users/Zohaibhassan.1/azureml-knn-maintenance")
df = pd.read_csv(os.path.join(repo, "data", "ai4i_clean.csv"))
NUM_COLS = ["air_temp_k", "process_temp_k", "rpm", "torque_nm", "tool_wear_min"]
CAT_COLS = ["type"]
df[NUM_COLS] = df[NUM_COLS].astype(float)
X, y = df[NUM_COLS + CAT_COLS], df["machine_failure"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
model = Pipeline([
    ("prep", ColumnTransformer([
        ("num", StandardScaler(), NUM_COLS),
        ("cat", OneHotEncoder(handle_unknown="ignore"), CAT_COLS),
    ])),
    ("knn", KNeighborsClassifier(n_neighbors=5)),
]).fit(X_train, y_train)
signature = infer_signature(X_train, model.predict(X_train))

# 2. Save locally (cloudpickle avoids the "untrusted types" error)
local_dir = os.path.expanduser("~/knn_model_local")
shutil.rmtree(local_dir, ignore_errors=True)
mlflow.sklearn.save_model(
    model, local_dir, signature=signature,
    serialization_format="cloudpickle",
    pip_requirements=[
        f"scikit-learn=={sklearn.__version__}",
        f"pandas=={pd.__version__}",
        f"numpy=={numpy.__version__}",
        "cloudpickle",
        f"mlflow=={mlflow.__version__}",
        "azureml-mlflow",
        "azureml-inference-server-http",
    ],
)

# 3. Register with the Azure ML SDK
registered = ml_client.models.create_or_update(Model(
    path=local_dir, type=AssetTypes.MLFLOW_MODEL,
    name="knn-maintenance-notebook", description="KNN maintenance, pinned packages",
))
print("Registered model version:", registered.version)

# 4. Make sure the endpoint exists
try:
    endpoint = ml_client.online_endpoints.get(ENDPOINT_NAME)
except Exception:
    ml_client.online_endpoints.begin_create_or_update(
        ManagedOnlineEndpoint(name=ENDPOINT_NAME, auth_mode="key")
    ).result()
    endpoint = ml_client.online_endpoints.get(ENDPOINT_NAME)

# 5. Remove any old deployment
try:
    ml_client.online_deployments.begin_delete(name="blue", endpoint_name=ENDPOINT_NAME).result()
except Exception:
    pass

# 6. Deploy (only small VM sizes, so quota is not exceeded)
deployed = False
for size in ["Standard_DS2_v2", "Standard_F2s_v2", "Standard_DS1_v2"]:
    print("Trying VM size:", size)
    try:
        ml_client.online_deployments.begin_create_or_update(ManagedOnlineDeployment(
            name="blue", endpoint_name=ENDPOINT_NAME, model=registered,
            instance_type=size, instance_count=1,
        )).result()
        deployed = True
        print("Deployed on", size)
        break
    except Exception as e:
        msg = str(e)
        print("Failed:", msg[:300])
        try:
            ml_client.online_deployments.begin_delete(name="blue", endpoint_name=ENDPOINT_NAME).result()
        except Exception:
            pass
        if not any(k in msg.lower() for k in ["quota", "sku", "core", "capacity"]):
            break

# 7. Send traffic and test
if deployed:
    endpoint = ml_client.online_endpoints.get(ENDPOINT_NAME)
    endpoint.traffic = {"blue": 100}
    ml_client.online_endpoints.begin_create_or_update(endpoint).result()

    sample = {"input_data": {
        "columns": ["air_temp_k", "process_temp_k", "rpm", "torque_nm", "tool_wear_min", "type"],
        "index": [0, 1],
        "data": [[298.1, 308.6, 1551.0, 42.8, 0.0, "M"],
                 [302.5, 310.9, 1325.0, 68.2, 215.0, "L"]],
    }}
    dep_dir = os.path.join(repo, "deployment")
    os.makedirs(dep_dir, exist_ok=True)
    req = os.path.join(dep_dir, "sample-request.json")
    json.dump(sample, open(req, "w"), indent=2)

    result = ml_client.online_endpoints.invoke(
        endpoint_name=ENDPOINT_NAME, deployment_name="blue", request_file=req)
    print("Prediction (0 = OK, 1 = failure):", result)
else:
    try:
        logs = ml_client.online_deployments.get_logs(name="blue", endpoint_name=ENDPOINT_NAME, lines=300)
        print("\n".join(l for l in logs.splitlines() if "Error" in l or "Traceback" in l))
    except Exception as e:
        print("No logs available:", str(e)[:200])

mlflow version: 2.22.5
Registered model version: 4
Trying VM size: Standard_DS2_v2
............................................................................................................................................................................................................................................Failed: (BadArgument) User container has crashed or terminated: Liveness probe failed: Get . Please see troubleshooting guide, available here: https://aka.ms/oe-tsg#error-resourcenotready
Code: BadArgument
Message: User container has crashed or terminated: Liveness probe failed: Get . Please see troubleshoo
No logs available: (BadRequest) The request is invalid.
Code: BadRequest
Message: The request is invalid.
Exception Details:	(InferencingClientCallFailed) Deployment /subscriptions/48b470e1-2c96-449b-8d68-d475b32d0a68/r


2026/10/05 14:33:31 WARNING mlflow.utils.requirements_utils: Detected one or more mismatches between the model's dependencies and the current Python environment:
 - azureml-inference-server-http (current: uninstalled, required: azureml-inference-server-http)
To fix the mismatches, call `mlflow.pyfunc.get_model_dependencies(model_uri)` to fetch the model's environment and install dependencies using the resulting environment file.
Uploading knn_model_local (0.73 MBs): 100%|██████████| 726215/726215 [00:00<00:00, 9400260.71it/s]




In [1]:
import os, json, shutil
import pandas as pd, numpy, sklearn, joblib
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier
from azure.ai.ml import MLClient
from azure.ai.ml.constants import AssetTypes
from azure.ai.ml.entities import (Model, Environment, CodeConfiguration,
                                  ManagedOnlineEndpoint, ManagedOnlineDeployment)
from azure.identity import DefaultAzureCredential

ml_client = MLClient(
    DefaultAzureCredential(),
    "48b470e1-2c96-449b-8d68-d475b32d0a68",
    "rg-mlab1-Zohaib-hassan",
    "mlw-lab1-Zohaib-hassan",
)
ENDPOINT_NAME = "knn-maint-zohaibhassan2"

# 1. Train the KNN model
repo = os.path.expanduser("~/cloudfiles/code/Users/Zohaibhassan.1/azureml-knn-maintenance")
df = pd.read_csv(os.path.join(repo, "data", "ai4i_clean.csv"))
NUM_COLS = ["air_temp_k", "process_temp_k", "rpm", "torque_nm", "tool_wear_min"]
CAT_COLS = ["type"]
df[NUM_COLS] = df[NUM_COLS].astype(float)
X, y = df[NUM_COLS + CAT_COLS], df["machine_failure"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)
model = Pipeline([
    ("prep", ColumnTransformer([
        ("num", StandardScaler(), NUM_COLS),
        ("cat", OneHotEncoder(handle_unknown="ignore"), CAT_COLS),
    ])),
    ("knn", KNeighborsClassifier(n_neighbors=5)),
]).fit(X_train, y_train)

# 2. Write the deployment files
base = os.path.expanduser("~/knn_deploy")
shutil.rmtree(base, ignore_errors=True)
for d in ["model", "code", "env"]:
    os.makedirs(os.path.join(base, d))
joblib.dump(model, os.path.join(base, "model", "model.pkl"))

with open(os.path.join(base, "code", "score.py"), "w") as f:
    f.write('''import os, json, joblib
import pandas as pd

NUM_COLS = ["air_temp_k", "process_temp_k", "rpm", "torque_nm", "tool_wear_min"]

def init():
    global model
    root = os.environ["AZUREML_MODEL_DIR"]
    path = None
    for folder, _, files in os.walk(root):
        if "model.pkl" in files:
            path = os.path.join(folder, "model.pkl")
    model = joblib.load(path)

def run(raw_data):
    data = json.loads(raw_data)["input_data"]
    frame = pd.DataFrame(data["data"], columns=data["columns"])
    frame[NUM_COLS] = frame[NUM_COLS].astype(float)
    return model.predict(frame).tolist()
''')

with open(os.path.join(base, "env", "conda.yaml"), "w") as f:
    f.write(f"""name: knn-env
channels:
  - conda-forge
dependencies:
  - python=3.10
  - pip
  - pip:
      - scikit-learn=={sklearn.__version__}
      - numpy=={numpy.__version__}
      - pandas=={pd.__version__}
      - joblib=={joblib.__version__}
      - azureml-inference-server-http
""")

# 3. Register the model and the environment
registered = ml_client.models.create_or_update(Model(
    path=os.path.join(base, "model"), type=AssetTypes.CUSTOM_MODEL,
    name="knn-maintenance-notebook", description="KNN maintenance (custom scoring)",
))
print("Registered model version:", registered.version)

env = Environment(
    name="knn-inference-env",
    image="mcr.microsoft.com/azureml/minimal-ubuntu22.04-py39-cpu-inference:latest",
    conda_file=os.path.join(base, "env", "conda.yaml"),
)
env = ml_client.environments.create_or_update(env)

# 4. Endpoint: create if missing, delete any old deployment
try:
    endpoint = ml_client.online_endpoints.get(ENDPOINT_NAME)
except Exception:
    ml_client.online_endpoints.begin_create_or_update(
        ManagedOnlineEndpoint(name=ENDPOINT_NAME, auth_mode="key")).result()
    endpoint = ml_client.online_endpoints.get(ENDPOINT_NAME)
try:
    ml_client.online_deployments.begin_delete(name="blue", endpoint_name=ENDPOINT_NAME).result()
except Exception:
    pass

# 5. Deploy (a failed deployment is KEPT so the logs can be read)
try:
    ml_client.online_deployments.begin_create_or_update(ManagedOnlineDeployment(
        name="blue", endpoint_name=ENDPOINT_NAME, model=registered, environment=env,
        code_configuration=CodeConfiguration(
            code=os.path.join(base, "code"), scoring_script="score.py"),
        instance_type="Standard_DS2_v2", instance_count=1,
    )).result()
    print("Deployment ready")
except Exception as e:
    print("Deployment failed:", str(e)[:400])
    try:
        logs = ml_client.online_deployments.get_logs(
            name="blue", endpoint_name=ENDPOINT_NAME, lines=300)
        print("\n".join(l for l in logs.splitlines()
                        if "Error" in l or "Traceback" in l or "No module" in l))
    except Exception as e2:
        print("No logs yet:", str(e2)[:200])
    raise

# 6. Traffic and test
endpoint = ml_client.online_endpoints.get(ENDPOINT_NAME)
endpoint.traffic = {"blue": 100}
ml_client.online_endpoints.begin_create_or_update(endpoint).result()

sample = {"input_data": {
    "columns": ["air_temp_k", "process_temp_k", "rpm", "torque_nm", "tool_wear_min", "type"],
    "index": [0, 1],
    "data": [[298.1, 308.6, 1551.0, 42.8, 0.0, "M"],
             [302.5, 310.9, 1325.0, 68.2, 215.0, "L"]],
}}
os.makedirs(os.path.join(repo, "deployment"), exist_ok=True)
req = os.path.join(repo, "deployment", "sample-request.json")
json.dump(sample, open(req, "w"), indent=2)
print("Prediction (0 = OK, 1 = failure):",
      ml_client.online_endpoints.invoke(
          endpoint_name=ENDPOINT_NAME, deployment_name="blue", request_file=req))

Uploading model (1.24 MBs): 100%|██████████| 1237704/1237704 [00:00<00:00, 28464389.20it/s]


Instance type Standard_DS2_v2 may be too small for compute resources. Minimum recommended compute SKU is Standard_DS3_v2 for general purpose endpoints. Learn more about SKUs here: https://learn.microsoft.com/azure/machine-learning/referencemanaged-online-endpoints-vm-sku-list
Check: endpoint knn-maint-zohaibhassan2 exists
Uploading code (0.0 MBs): 100%|██████████| 597/597 [00:00<00:00, 63422.90it/s]




Registered model version: 5
............................................................................................................................................................Deployment ready
Prediction (0 = OK, 1 = failure): [0, 1]


In [7]:
import json, os

sample = {
    "input_data": {
        "columns": ["air_temp_k", "process_temp_k", "rpm", "torque_nm", "tool_wear_min", "type"],
        "index": [0, 1],
        "data": [
            [298.1, 308.6, 1551.0, 42.8, 0.0, "M"],
            [302.5, 310.9, 1325.0, 68.2, 215.0, "L"],
        ],
    }
}

dep_dir = os.path.expanduser("~/cloudfiles/code/Users/Zohaibhassan.1/azureml-knn-maintenance/deployment")
os.makedirs(dep_dir, exist_ok=True)
req_path = os.path.join(dep_dir, "sample-request.json")

with open(req_path, "w") as f:
    json.dump(sample, f, indent=2)

print(json.dumps(sample, indent=2))
print("Saved to:", req_path)

{
  "input_data": {
    "columns": [
      "air_temp_k",
      "process_temp_k",
      "rpm",
      "torque_nm",
      "tool_wear_min",
      "type"
    ],
    "index": [
      0,
      1
    ],
    "data": [
      [
        298.1,
        308.6,
        1551.0,
        42.8,
        0.0,
        "M"
      ],
      [
        302.5,
        310.9,
        1325.0,
        68.2,
        215.0,
        "L"
      ]
    ]
  }
}
Saved to: /home/azureuser/cloudfiles/code/Users/Zohaibhassan.1/azureml-knn-maintenance/deployment/sample-request.json


In [9]:
ENDPOINT_NAME = "knn-maint-zohaibhassan2"
req_path = os.path.expanduser(
    "~/cloudfiles/code/Users/Zohaibhassan.1/azureml-knn-maintenance/deployment/sample-request.json"
)
print("Request file exists:", os.path.isfile(req_path))

response = ml_client.online_endpoints.invoke(
    endpoint_name=ENDPOINT_NAME,
    deployment_name="blue",
    request_file=req_path,
)
print("Prediction (0 = OK, 1 = failure):", response)

Request file exists: True
Prediction (0 = OK, 1 = failure): [0, 1]
